# Ghana-LLM — train + instruction-tune on Colab GPU

From-scratch Ghanaian / Black-history LLM. **Code** from GitHub (public); **data + checkpoints** on *your* Google Drive (corpus never in git).

Dataset **`gh_v4`**: Bible (5 langs) + Ghanaian Wikipedia + 120 curated history books (~20.9M tokens). Then an **SFT** stage turns the base model into a prompt→answer assistant (translation + Ghana/Black-history Q&A).

### First
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Upload `ghana-llm-data.tar.gz` to Drive at **`MyDrive/ghana-llm/`**.
3. **Runtime → Run all.**

In [ ]:
#@title 1) Check the GPU
import torch, subprocess
print(subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout)
print("CUDA:", torch.cuda.is_available(), "| torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU"

In [ ]:
#@title 2) Pull the CODE from GitHub (public — no token)
import os, shutil
if os.path.isdir("ghana-llm"): shutil.rmtree("ghana-llm")
!git clone --depth 1 https://github.com/Eselase-Noble/ghana-llm.git
%cd ghana-llm
!pip install -q -r requirements-colab.txt
print("code ready")

In [ ]:
#@title 3) Mount Drive, unpack data, link it in
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ghana-llm'
import os, shutil, subprocess
if not os.path.isdir(f'{DRIVE}/corpus/tokenized/gh_v4'):
    assert os.path.exists(f'{DRIVE}/ghana-llm-data.tar.gz'), 'Upload ghana-llm-data.tar.gz to MyDrive/ghana-llm/ first'
    subprocess.run(['tar','xzf',f'{DRIVE}/ghana-llm-data.tar.gz','-C',DRIVE], check=True)
if os.path.islink('corpus'): os.remove('corpus')
elif os.path.isdir('corpus'): shutil.rmtree('corpus')
os.symlink(f'{DRIVE}/corpus', 'corpus')
CKPT_DIR = f'{DRIVE}/checkpoints/gh_v4'; os.makedirs(CKPT_DIR, exist_ok=True)
assert os.path.exists('corpus/tokenized/gh_v4/train.bin'), 'data not found in bundle'
print('data + checkpoints ready at', DRIVE)

## Stage A — pretrain the base model (text understanding)

In [ ]:
#@title 4) Train the BASE model on GPU  (RESUMABLE: re-run to continue)
!python src/train/train.py \
    --config configs/gpu.json \
    --dataset gh_v4 \
    --device cuda \
    --ckpt-dir "$CKPT_DIR" \
    --resume

In [ ]:
#@title 5) (optional) sample raw text from the base model
!python src/generate/sample.py --dataset gh_v4 --device cuda \
    --ckpt "$CKPT_DIR/ckpt.pt" --prompt "The history of Ghana" --tokens 150 --temperature 0.8 --top-k 50

## Stage B — instruction-tune (prompt → answer)
Turns the base model into an assistant using translation pairs + Ghana/Black-history Q&A.

In [ ]:
#@title 6) SFT: fine-tune into a question-answering assistant
!python src/train/sft.py \
    --dataset gh_v4 \
    --base "$CKPT_DIR/ckpt.pt" \
    --out "$CKPT_DIR/sft.pt" \
    --device cuda \
    --epochs 3

In [ ]:
#@title 7) Ask it a question
PROMPT = "Who was Kwame Nkrumah?"  #@param {type:"string"}
!python src/generate/chat.py --dataset gh_v4 --device cuda \
    --ckpt "$CKPT_DIR/sft.pt" --prompt "$PROMPT"
# try also: 'Translate to Twi: Ghana is my country'  |  'What was the Ashanti Empire?'

---
## (Optional) Grow the corpus on Colab, then re-pretrain + re-SFT
```
!python src/scrape/black_history.py
!python src/scrape/books.py --limit-per-query 25
!python src/data/build_corpus.py
!python src/data/prepare.py --dataset gh_v5 --glob '*.txt' --vocab-size 16000 --retrain-tokenizer
```
Then train with `--dataset gh_v5`. For a bigger model, use `configs/base_gpu.json`.